In [15]:
import ee
import geemap

# ========================
# 0. Initialize GEE (Google Earth Engine)
# ========================
# Note: Replace the project ID with your own GEE project ID before running
ee.Initialize(project='YOUR_GEE_PROJECT_ID')

# ========================
# 1. Get administrative boundary of Beijing
# ========================
# GAUL (Global Administrative Unit Layers) dataset - Level 1 administrative divisions of China
china_adm1 = ee.FeatureCollection("FAO/GAUL/2015/level1")

# Filter boundary by administrative name (GAUL dataset uses "Beijing Shi" for Beijing)
beijing_boundary = china_adm1.filter(
    ee.Filter.eq('ADM1_NAME', 'Beijing Shi')
).geometry()

# Define the study region
region = beijing_boundary

# ========================
# 2. Sentinel-2 data preprocessing
# ========================
def maskS2clouds(image):
    """
    Mask cloud and cirrus pixels in Sentinel-2 SR images
    Args:
        image: ee.Image - Sentinel-2 image
    Returns:
        ee.Image - Cloud-masked image with values scaled to [0,1]
    """
    qa = image.select('QA60')
    cloudBitMask = 1 << 10  # Cloud bit (10)
    cirrusBitMask = 1 << 11 # Cirrus bit (11)
    # Create mask (0 = cloud/cirrus, 1 = clear)
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
        qa.bitwiseAnd(cirrusBitMask).eq(0)
    )
    return image.updateMask(mask).divide(10000)

# Load Sentinel-2 Surface Reflectance Harmonized dataset
s2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate('2024-01-01', '2024-12-31') \
    .filterBounds(region) \
    .map(maskS2clouds)

# Calculate median composite to reduce temporal noise
s2_median = s2_collection.median().clip(region)

# ========================
# 3. Spectral index calculation
# ========================
# NDVI (Normalized Difference Vegetation Index)
ndvi = s2_median.normalizedDifference(['B8', 'B4']).rename('NDVI')
# NDBI (Normalized Difference Built-up Index)
ndbi = s2_median.normalizedDifference(['B11', 'B8']).rename('NDBI')
# MNDWI (Modified Normalized Difference Water Index)
mndwi = s2_median.normalizedDifference(['B3', 'B11']).rename('MNDWI')
# SWIR/NIR ratio
swir_nir_ratio = s2_median.select('B11').divide(s2_median.select('B8')).rename('SWIR_NIR')

# ========================
# 4. Texture features (GLCM - Gray Level Co-occurrence Matrix)
# ========================
# Convert NIR band (B8) to 8-bit for GLCM calculation
glcm = s2_median.select('B8').multiply(255).toUint8().glcmTexture(size=3)

# Select key texture features and rename for readability
texture_features = glcm.select([
    'B8_contrast',
    'B8_ent',
    'B8_var',
    'B8_idm',
    'B8_diss'
]).rename([
    'Contrast',
    'Entropy',
    'Variance',
    'Homogeneity',
    'Dissimilarity'
])

# ========================
# 5. Nighttime light data (VIIRS)
# ========================
# Load VIIRS monthly nighttime light dataset (2024-2025)
viirs_2024 = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG") \
    .filterDate('2024-01-01', '2025-11-01') \
    .select('avg_rad') \
    .mean() \
    .clip(region) \
    .rename('Nightlight')

# ========================
# 6. Merge all features into a single image
# ========================
feature_image = ee.Image.cat([
    s2_median.select(['B2','B3','B4','B8','B11','B12']),  # Optical bands
    ndvi, ndbi, mndwi, swir_nir_ratio,                   # Spectral indices
    texture_features,                                    # Texture features
    viirs_2024                                           # Nighttime light
]).clip(region)

# Print band names for verification
print("✅ Final feature image bands:", feature_image.bandNames().getInfo())

# ========================
# 7. Visualization (for verification)
# ========================
Map = geemap.Map()
Map.addLayer(region, {'color': 'red'}, "Beijing Boundary")
Map.addLayer(feature_image, {
    'bands': ['B4','B3','B2'],
    'min': 0, 'max': 0.3
}, "Beijing Composite")
Map.centerObject(region, 8)
Map
# ========================
# 8. Export feature image to GEE Asset
# ========================
# Note: Replace "YOUR_GEE_PROJECT_ID" with your actual GEE project ID
task_asset = ee.batch.Export.image.toAsset(
    image=feature_image,
    description='Beijing_Feature_Image',
    assetId='projects/YOUR_GEE_PROJECT_ID/assets/Beijing_Feature_Image',
    region=region,
    scale=10,       
    maxPixels=1e13
)

task_asset.start()

# Convert image to float type for more stable export (optional but recommended)
feature_image = feature_image.toFloat()

# ========================
# 9. Export feature image to Google Drive
# ========================
# 30m scale is recommended for more stable export (10m may cause memory issues for large regions)
task_drive = ee.batch.Export.image.toDrive(
    image=feature_image,
    description='Beijing_features_toDrive',
    folder='GEE_exports',
    fileNamePrefix='Beijing_features_V1',
    scale=10,
    region=region,
    maxPixels=1e13
)

task_drive.start()

# Print export task initiation confirmation
print(" Export tasks initiated successfully. Check GEE Tasks interface for progress.")

🚀 ROC验证数据导出任务已启动，请到 GEE Tasks 查看进度。


In [5]:
import ee
import geemap
import pandas as pd

    ee.Initialize(
        project='',
        opt_url='https://earthengine-highvolume.googleapis.com'

feature_image = ee.Image('projects/'YOUR_GEE_PROJECT_ID'/assets/Beijing_Feature_V1')


csv_path_positive = r"'YOUR_Positive_sample_path'"

    poi_fc = geemap.csv_to_ee(csv_path_positive, latitude='latitude', longitude='longitude')
    positive_samples = poi_fc.map(lambda f: f.set('label', 1))
    num_positive = positive_samples.size().getInfo()


china_adm1 = ee.FeatureCollection("FAO/GAUL/2015/level1")
beijing_boundary = china_adm1.filter(ee.Filter.eq('ADM1_NAME', 'Beijing Shi')).geometry()

negative_samples = ee.FeatureCollection([])
if num_positive > 0:
    negative_points = ee.FeatureCollection.randomPoints(beijing_boundary, 1000, seed=42)
    negative_samples = negative_points.map(lambda f: f.set('label', 0))
    negative_samples = negative_samples.randomColumn('rand', seed=42).limit(num_positive)
    print(f"✅ negative_samples: {negative_samples.size().getInfo()}")


all_samples = positive_samples.merge(negative_samples)
training_points = validation_points = ee.FeatureCollection([])
if all_samples.size().getInfo() > 1:
    all_samples_random = all_samples.randomColumn('random', seed=42)
    training_points = all_samples_random.filter(ee.Filter.lt('random', 0.7))
    validation_points = all_samples_random.filter(ee.Filter.gte('random', 0.7))
    print(f"📚 training_points: {training_points.size().getInfo()} | validation_points: {validation_points.size().getInfo()}")


classifier = None
prob_image = None
if training_points.size().getInfo() > 0:
    print("\n🚀 Train a random forest (probability mode)...")
    training_data = feature_image.sampleRegions(
        collection=training_points, 
        properties=['label'], 
        scale=10, 
        tileScale=4
    )

    classifier = ee.Classifier.smileRandomForest(numberOfTrees=100, seed=42).train(
        features=training_data,
        classProperty='label'
    ).setOutputMode('PROBABILITY')
    print("✅Model training completed (probability mode)")


    prob_image = feature_image.classify(classifier).clip(beijing_boundary)
    prob_image = prob_image.rename('probability').toFloat() 
    print(f"✅ Probability image generation successful, band type: {prob_image.getInfo()['bands'][0]['data_type']['type']}")

  
    if validation_points.size().getInfo() > 0:
        val_classifier = classifier.setOutputMode('CLASSIFICATION')
        validation_data = feature_image.sampleRegions(collection=validation_points, properties=['label'], scale=10)
        validation_classified = validation_data.classify(val_classifier)
        cm = validation_classified.errorMatrix('label', 'classification')
        cm_info = cm.getInfo()
        print("\n📊 Accuracy Evaluation:")
        print("confusion matrix:", cm_info)
        print(f"Overall accuracy rate: {cm.accuracy().getInfo():.4f}")
        TN, FP, FN, TP = cm_info[0][0], cm_info[0][1], cm_info[1][0], cm_info[1][1]
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        print(f"Precision={precision:.4f}, Recall={recall:.4f}, F1={f1_score:.4f}")


if prob_image is not None:
  
    task_asset = ee.batch.Export.image.toAsset(
        image=prob_image,
        description='Beijing_Probability_RF_Revised1',
        assetId='projects/'YOUR_GEE_PROJECT_ID'/assets/Beijing_Probability_RF_Revised1',
        region=beijing_boundary,
        scale=10,
        maxPixels=1e13
    )
    task_asset.start()
    
    
    task_drive = ee.batch.Export.image.toDrive(
        image=prob_image,
        description='Beijing_Probability_RF',
        folder='GEE_exports',
        fileNamePrefix='Beijing_Probability',
        region=beijing_boundary,
        scale=10,
        maxPixels=1e13,
        fileFormat='GeoTIFF'  
        
    )
    task_drive.start()


if prob_image is not None:
    Map = geemap.Map()
    Map.centerObject(beijing_boundary, 10)
 
    Map.addLayer(prob_image, {
        'min': 0, 
        'max': 1, 
        'palette': ['#0000FF', '#FFFF00', '#FF0000']
    }, "Beijing Probability (0→1)")
    Map.addLayer(positive_samples, {'color':'green', 'pointRadius':6}, "True Data Centers")
    Map.addLayerControl()
    Map

✅ GEE 初始化成功
✅ 北京正样本数量: 83
✅ 负样本数量: 83
📚 训练集: 117 | 验证集: 49

🚀 训练随机森林（概率模式）...
✅ 模型训练完成（概率模式）
✅ 概率影像生成成功，波段类型: PixelType

📊 精度评估:
混淆矩阵: [[21, 4], [4, 20]]
总体准确率: 0.8367
Precision=0.8333, Recall=0.8333, F1=0.8333

🚀 概率影像导出到Asset，任务启动！
🚀 概率影像导出到Google Drive，任务启动！
